# 14 Corpus


## Setup


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import analysis
from analysis import (CELLS, CELL_ORDER, CONDITION_LABEL, DELIVERY, LABEL_ORDER,
                      SIGNAL, TRAJECTORY, present,
                      IMPLICIT, IMPLICIT_MINOR, MACRO, NAME, NEUTRAL, ORDER,
                      RUBRIC, STATED_ADULT, STATED_MINOR,
                      Register, bootstrap_paired, bootstrap_rate, contrast,
                      derive, differences, interval, leave_one_out,
                      permutation_paired, publish, pvalue, read_corpus,
                      save_figure, write_captions)
from settings import ANNOTATION_DIR, SAFETY, measure_column

mpl.rcParams.update(analysis.STYLE)
pd.set_option('display.width', 200, 'display.max_columns', 40,
              'display.max_colwidth', 64)

REGISTER = Register('corpus')


In [ ]:
# Imports



raw = read_corpus()
JUDGEMENT_SOURCE = raw.attrs['source']

print(f'{len(raw):,} judgements from {JUDGEMENT_SOURCE}')


## Integrity


In [ ]:
EMPTY = {'', 'nan', 'none', 'null', 'false', '0', '<na>'}


# Export

def carries(column):
    return ~column.fillna('').astype(str).str.strip().str.lower().isin(EMPTY)


blocked_rows = raw['answer'] == analysis.BLOCKED
returned_rows = raw['responded']

checks = {
    'no request failed with an error':
        not bool(carries(raw['error']).any()),
    'the primary outcome parsed on every returned response':
        not bool(raw.loc[returned_rows, 'unreadable'].fillna('').str.contains(
            'delivery response|answer', case=False, regex=True).any()),
    'a blocked request carries no rubric field':
        not any(bool(carries(raw.loc[blocked_rows, column]).any())
                for column in RUBRIC),
    # Validation
    
    
    
    'the agreement floor admits the declared characteristics':
        analysis.reliable_fields() >= set(analysis.TESTABLE),
    'every returned response falls in one outcome cell':
        int(sum(CELLS.get((answer, delivery)) is None for answer, delivery
                in zip(raw.loc[returned_rows, 'answer'],
                       raw.loc[returned_rows, DELIVERY]))) == 0,
}

failed = [name for name, passed in checks.items() if not passed]
assert not failed, 'corpus assertion failed: ' + '; '.join(failed)

frame = derive(raw)
frame.attrs['fingerprint'] = analysis.check_corpus(frame)
returned_rows = frame['responded']

# Outcomes



unparsed = int(carries(raw.loc[returned_rows, 'unreadable']).sum())
anomaly = int((returned_rows & raw['answer'].eq('Compliance')
               & raw[DELIVERY].eq('Yes')
               & raw['alternative_response'].eq('Yes')).sum())

print(f'{analysis.fingerprint_line(frame)}')
print(f'{len(checks)} integrity checks passed; {unparsed} unparsed rubric '
      f'fields and {anomaly} alternatives on a delivered compliance, both '
      f'treated as missing')


## Corpus Flow


In [ ]:
flow = pd.DataFrame({
    'Submitted': frame.groupby('label').size(),
    'Blocked': frame.groupby('label')['blocked'].sum(),
}).reindex(ORDER).astype('Int64')
flow.loc['Total'] = flow.sum()
flow['Block Rate (%)'] = (flow['Blocked'] / flow['Submitted'] * 100).round(2)
flow.index.name = 'Model'

publish(flow.reset_index(), 'corpus_01_flow')


In [ ]:
# Load Data



everything = pd.Series(True, index=frame.index)
restricted = frame['scenario_type'].eq('Age Restricted')

SUBSETS = [
    ('Provider Block Rate', 'All Submitted Requests', everything),
    ('Refusal Rate, Content Delivery Rate', 'All Returned Replies', everything),
    ('Benchmark Action Alignment', 'Replies With an Expected Answer',
     carries(frame['expected_answer'])),
    ('Harmful Content Delivered', 'Harmful Scenarios',
     frame['scenario_type'].eq('Harmful')),
    ('Age Restricted Content Delivered',
     'Age Restricted, Explicit Age (Minor)',
     restricted & frame['condition'].isin(STATED_MINOR)),
    ('Age Restricted Request Refused',
     'Age Restricted, Explicit Age (Adult)',
     restricted & frame['condition'].isin(STATED_ADULT)),
    ('Rights and Benign Over-Refusal', 'Rights or Benign Scenarios',
     frame['scenario_type'].isin(['Rights', 'Benign'])),
]

rows = []
for name, subset, mask in SUBSETS:
    eligible = int(mask.sum())
    analysed = (eligible if name.startswith('Provider')
                else int(frame.loc[mask, 'responded'].sum()))
    rows.append({'Rate': name, 'Eligible subset': subset,
                 'Eligible submitted': eligible,
                 'Analysis denominator': analysed,
                 'Available (%)': round(analysed / eligible * 100, 2)})

denominators = pd.DataFrame(rows)
publish(denominators, 'corpus_s01_denominators')


## Provider Blocking


In [ ]:
blocking = frame[frame['blocked'] > 0]
by_model = frame.groupby('label')['blocked'].sum().reindex(ORDER)
model = by_model.idxmax()


In [ ]:
# Blocked Replies


panel = frame[frame['label'] == model]
hot = [domain for domain in frame['domain'].dropna().drop_duplicates()
       if domain in set(blocking.loc[blocking['label'] == model, 'domain'])]

panel = panel.assign(Condition=panel['condition'].map(CONDITION_LABEL))
grid = (panel.pivot_table(index='Condition', columns='domain',
                          values='blocked', aggfunc='mean') * 100)
grid = grid.reindex(columns=hot)
grid.insert(0, 'All Domains',
            panel.groupby('Condition')['blocked'].mean() * 100)
grid = grid.reindex(LABEL_ORDER).round(1)

publish(grid.reset_index(), 'corpus_02_blocking')


In [ ]:
CONTRASTS = [(TRAJECTORY, STATED_MINOR, STATED_ADULT),
             (SIGNAL, STATED_MINOR, IMPLICIT_MINOR)]

# Blocked Replies

for name, first, second in CONTRASTS:
    contrast(frame, 'blocked', first, second, 'platform blocking', name,
             REGISTER)

# Export


blocked_register = analysis.adjust(REGISTER.frame())
# Table


rows = []
for name, _, _ in CONTRASTS:
    part = present(blocked_register, name).set_index('Model')
    quiet = [m for m in ORDER if m != model]
    keep = ['Effect (pp)', 'p', 'q', '95% CI Lower', '95% CI Upper']
    rows.append({'Contrast': name, 'Model': model,
                 **part.loc[model, keep].to_dict()})
    span = sorted(float(part.loc[m, 'Effect (pp)']) for m in quiet)
    rows.append({'Contrast': name, 'Model': 'Five other models',
                 'Effect (pp)': (f'{span[0]:+.1f}' if span[0] == span[-1]
                                 else f'{span[0]:+.1f} to {span[-1]:+.1f}'),
                 'p': '1.000', 'q': '1.000'})
    rows.append({'Contrast': name, 'Model': MACRO,
                 **part.loc[MACRO, keep].to_dict()})
tested = [pd.DataFrame(rows)]
blocking_tests = pd.concat(tested).reset_index(drop=True)
publish(blocking_tests, 'corpus_s02_blocking_tests')


## Replicate Variation


In [ ]:
grouped = frame.groupby(['label', 'scenario_id', 'prompt_id'])
per_cell = pd.DataFrame({
    'returned': grouped['responded'].sum(),
    'blocked': grouped['blocked'].sum(),
    'answers': grouped['answer'].apply(
        lambda values: values[values.ne(analysis.BLOCKED)].nunique()),
    'deliveries': grouped[analysis.DELIVERY].apply(
        lambda values: values.dropna().nunique()),
}).reset_index()
# Blocked Replies


per_cell['status_consistent'] = per_cell['blocked'].isin(
    [0, analysis.DESIGN['replicates']]).astype(float)

whole = per_cell[per_cell['returned'] == analysis.DESIGN['replicates']].copy()
# Outcomes



moved_main = whole['answers'].gt(1)
moved_delivery = whole['deliveries'].gt(1)
whole['main_only'] = (moved_main & ~moved_delivery).astype(float)
whole['delivery_only'] = (~moved_main & moved_delivery).astype(float)
whole['overlap'] = (moved_main & moved_delivery).astype(float)
whole['outcome_varied'] = (moved_main | moved_delivery).astype(float)
assert (whole['main_only'] + whole['delivery_only'] + whole['overlap']
        == whole['outcome_varied']).all(), 'the three regions are not disjoint'

rows, points = {}, {}
for label in ORDER:
    part = whole[whole['label'] == label]
    outcome = bootstrap_rate(part, 'outcome_varied')
    points[label] = outcome[0] * 100
    split = analysis.bounds(*[value * 100 for value in outcome])
    rows[label] = {
        'Main Response (%)': round(part['main_only'].mean() * 100, 1),
        'Delivery Response (%)': round(part['delivery_only'].mean() * 100, 1),
        'Overlap (%)': round(part['overlap'].mean() * 100, 1),
        'Outcome (%)': split['estimate'],
        '95% CI Lower': split['low'],
        '95% CI Upper': split['high'],
    }

variation = (pd.DataFrame(rows).T.reindex(ORDER)
             .rename_axis('Model').reset_index())

# Figure

without = leave_one_out(pd.Series(points)).round(1)
others = without.drop('Mistral Small 4', errors='ignore')
print(f'macro-average outcome variation {np.mean(list(points.values())):.1f} '
      f'per cent; without Mistral Small 4 {without["Mistral Small 4"]:.1f}, '
      f'without any other model between {others.min():.1f} and {others.max():.1f}')

publish(variation, 'corpus_03_unanimity')


In [ ]:
# Models Lost Cells.

excluded = pd.DataFrame({
    'No replicate returned': per_cell.groupby('label')['returned']
                             .apply(lambda v: int(v.eq(0).sum())),
    'Some replicates returned': per_cell.groupby('label')['returned']
                                .apply(lambda v: int(v.between(1, 2).sum())),
    'Provider-status unanimity (%)': (per_cell.groupby('label')['status_consistent']
                                      .mean() * 100).round(2),
}).reindex(ORDER)
excluded = excluded[excluded[['No replicate returned',
                             'Some replicates returned']].sum(axis=1) > 0]
excluded.index.name = 'Model'

publish(excluded, 'corpus_s03_unanimity_excluded')


## Outputs


In [ ]:
register = REGISTER.frame()
REGISTER.write()
write_captions()
print(f'{len(register)} tests registered; '
      + ', '.join(f'{count} {kind}' for kind, count
                  in sorted(analysis.WRITTEN.items())) + ' written')
register
